# Saved online encoder versus EMA teacher diagnostic

Purpose: determine whether the saved 500-update online encoders outperform the slowly updated teachers enough to select faster EMA as the next exploratory mathematical factor. This notebook exports reusable definitions; the Notebook 04 controller owns the bounded GPU run. Running this module top-to-bottom only loads its definitions.

Inputs: immutable `semantic-local-pilot-002` final ViT and semantic checkpoints, the verified Food101 training/development split (10,100 / 2,020 images), existing teacher feature caches and authoritative final epoch-20 probe checkpoints. Official test data are never decoded or evaluated.

Outputs: `artifact_dir/saved_encoder_diagnostics/{protocol,manifest,result}.json`, one online probe summary per architecture, content-addressed online feature caches, and a Markdown diagnostic report. Expected runtime: approximately 3–5 minutes on GPU 0 plus CPU probes; the caller's absolute deadline is checked during feature extraction and each 50 probe updates. A hard wall-clock guard belongs to the detached controller.

Checkpoints: `checkpoints/<artifact_dir.name>/saved_encoder_probes/<kind>_online/`, preserving latest two valid probe checkpoints and the best using the existing audited helper. No encoder training occurs.

Resume exactly: the controller reimports only `ahr-export` cells and calls `run_saved_encoder_diagnostics(ROOT, identical_artifact_dir, original_deadline_unix)`. Full input and source identities must match; invalid caches or checkpoints raise. Do not extend the registered deadline without authorization.

Predeclared screen: if either semantic or ViT online-minus-teacher development top-1 point difference is at least **+0.5 percentage points**, select EMA start 0.99 (instead of 0.996), rising to 1, as the next mathematical factor. Otherwise select semantic feature count 32 versus 64, alongside a ViT reference. Both representations are evaluated at fixed final probe epoch 20; the prior best-development-epoch readout is not used for this decision. This is an exploratory selection rule, not a significance test. Paired image bootstrap intervals exclude training-seed and diagnostic-driven factor-selection uncertainty.


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat
ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_boot_path = ROOT / "fmm_jepa/bootstrap.ipynb"
_boot_ns = {"__file__": str(_boot_path)}
for _cell in nbformat.read(_boot_path, as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, str(_boot_path), "exec"), _boot_ns)
_boot_ns["install_notebook_package"](ROOT)


In [ ]:
from pathlib import Path
import copy, csv, gc, json, math, time
import numpy as np
import torch
from torch import nn
from fmm_jepa.training import atomic_json, digest_file, definition_hashes, run_writer
from fmm_jepa.food101_training import validate_training_manifest, isolated_rng
from fmm_jepa.food101_data import EvaluationDataset, preprocessing_descriptor
from fmm_jepa.food101_evaluation import (encoder_state_hash, stable_digest, extract_features,
    validate_feature_cache, fit_linear_probe, fit_standardization, standardize,
    classification_metrics, paired_bootstrap, write_csv, _probe_config, _load_probe_checkpoint, prediction_rows)
from fmm_jepa.semantic_evaluation import (_build_semantic_comparator, _DeadlineDataset,
    _deadline_check, _release_semantic)
from fmm_jepa.semantic_pilot import reuse_food_bundle, configure_environment

SOURCE_RUN = "semantic-local-pilot-002"
DIAGNOSTIC_KINDS = ("vit", "semantic")
DECISION_RULE = {
    "threshold_percentage_points": 0.5,
    "probe_reporting_selection": "fixed_final_epoch_20",
    "criterion": "semantic online-minus-teacher top1 >= 0.5 pp OR ViT online-minus-teacher top1 >= 0.5 pp",
    "if_triggered": "ema_start_0.99_vs_0.996_both_rising_to_1",
    "otherwise": "semantic_features_32_vs_64_with_vit_reference",
    "interpretation": "Exploratory point-estimate screen, not statistical significance; frozen before extracting online features."
}

def _diagnostic_source_hash(root):
    path = Path(root) / "experiments/04_followup/diagnostics.ipynb"
    document = json.loads(path.read_text())
    sources = ["".join(cell["source"]) for cell in document["cells"]
               if cell["cell_type"] == "code" and "ahr-export" in cell.get("metadata", {}).get("tags", [])]
    return stable_digest(sources)

def _diagnostic_probe(train_cache, dev_cache, checkpoint_dir, config, model_id, encoder, deadline):
    """Reuse the audited full-state probe continuation in chunks of 50 updates."""
    folder = Path(checkpoint_dir)
    while True:
        _deadline_check(deadline, reserve=2)
        path = folder / model_id / "probe_manifest.json"
        updates = int(json.loads(path.read_text())["latest"]["successful_updates"]) if path.exists() else 0
        result = fit_linear_probe(train_cache, dev_cache, folder, config, model_id,
                                  encoder=encoder, stop_after_updates=updates + 50)
        manifest = json.loads(path.read_text())
        manifest["resume_notebook"] = "experiments/04_followup/diagnostics.ipynb"
        manifest["resume_cell"] = "run_saved_encoder_diagnostics with the same artifact directory and original deadline"
        atomic_json(path, manifest)
        if result.get("status") != "INTERRUPTED":
            return result

def _fixed_final_epoch_probe(probe, caches, checkpoint_folder):
    """Read the validated latest probe state, never choose an epoch from dev quality."""
    original = probe["summary"]
    selected = torch.load(original["selected_probe_path"], map_location="cpu", weights_only=True)
    payload = _load_probe_checkpoint(Path(checkpoint_folder), selected["identity"])
    if payload is None or int(payload["completed_epochs"]) != 20 or int(payload["next_offset"]) != 0 or payload["active_order"] is not None:
        raise ValueError("Fixed final probe requires a fully completed epoch-20 checkpoint")
    if int(payload["successful_updates"]) != 800:
        raise ValueError("Final epoch-20 probe update budget is not 800")
    if payload["initialization_hash"] != original["initialization_hash"]:
        raise ValueError("Latest probe initialization identity differs")
    if selected["identity"]["train_cache"] != caches["train"]["content_hash"] or selected["identity"]["dev_cache"] != caches["development"]["content_hash"]:
        raise ValueError("Final probe caches differ from training identity")
    stats = fit_standardization(caches["train"]["features"], original["budget"]["standardize"])
    if any(not torch.equal(stats[key], payload["standardization"][key]) for key in ("mean", "std")):
        raise ValueError("Final probe standardization does not reconstruct from training data")
    classifier = nn.Linear(caches["train"]["features"].shape[1], 101)
    classifier.load_state_dict(payload["classifier"])
    classifier.eval()
    with torch.inference_mode():
        features = standardize(caches["development"]["features"], stats)
        logits = torch.cat([classifier(x) for x in features.split(original["budget"]["batch_size"])])
    metrics = classification_metrics(logits, caches["development"]["labels"])
    recorded = [row for row in payload["metrics"] if row["split"] == "development" and int(row["epoch"]) == 20]
    if len(recorded) != 1 or any(abs(metrics[key] - recorded[0][key]) > 1e-7 for key in ("top1", "top5", "loss")):
        raise ValueError("Final epoch-20 metrics do not reconstruct from the checkpoint")
    state_hash = encoder_state_hash(payload["classifier"])
    rows = prediction_rows(logits, caches["development"]["labels"], caches["development"]["sample_ids"],
        original["model_id"], "development", original["checkpoint_hash"], state_hash, original["seed"])
    manifest_path = Path(checkpoint_folder) / "probe_manifest.json"
    manifest = json.loads(manifest_path.read_text())
    latest_record = manifest["latest"]
    checkpoint_file = latest_record.get("file", latest_record.get("path"))
    if checkpoint_file is None:
        raise ValueError("Unknown final probe checkpoint manifest path field")
    checkpoint_path = Path(checkpoint_folder) / checkpoint_file
    summary = {**copy.deepcopy(original), "selection": "fixed_final_epoch", "selected_epoch": 20,
        "metrics": metrics, "probe_hash": state_hash,
        "prior_selection_path_for_identity_only": original["selected_probe_path"],
        "reported_checkpoint_path": str(checkpoint_path), "reported_checkpoint_sha256": digest_file(checkpoint_path),
        "reporting_note": "Fixed final epoch 20; legacy selected_probe_path was used only to validate checkpoint identity, not to select the reported classifier."}
    return {"summary": summary, "metrics": copy.deepcopy(payload["metrics"]), "predictions": rows}

def _load_verified_teacher(root, source, config, bundle, kind, teacher):
    """Read and independently reconstruct the previously selected teacher probe on CPU."""
    evaluation = json.loads((source / "semantic_evaluation_manifest.json").read_text())
    model_id = kind + "_trained"
    summary_path = source / (model_id + "_semantic_probe.json")
    entry = evaluation["completed"][model_id]
    if digest_file(summary_path) != entry["sha256"]:
        raise ValueError("Teacher summary differs from original evaluation manifest")
    saved = json.loads(summary_path.read_text())
    if saved["identity_hash"] != evaluation["identity_hash"]:
        raise ValueError("Teacher evaluation identity mismatch")
    probe = saved["probe"]
    common = {"dataset_revision": bundle["dataset"].fingerprint,
              "checkpoint_hash": encoder_state_hash(teacher), "encoder_kind": kind,
              "preprocessing": preprocessing_descriptor(config),
              "pooling": "mean_final_layernorm_patches",
              "normalization": "encoder final affine LayerNorm then patch mean; train-only per-feature probe standardization"}
    caches, cache_evidence = {}, {}
    for split, ids in (("train", bundle["train_ids"]), ("development", bundle["dev_ids"])):
        identity = {**common, "sample_ids": list(ids)}
        path = source / "semantic_features" / model_id / (stable_digest(identity) + ".pt")
        meta = json.loads(path.with_suffix(".json").read_text())
        if meta["sha256"] != digest_file(path):
            raise ValueError("Teacher cache file hash changed")
        cache = validate_feature_cache(torch.load(path, map_location="cpu", weights_only=True), identity)
        caches[split] = {**cache, "path": str(path)}
        cache_evidence[split] = {"path": str(path), "sha256": meta["sha256"], "content_hash": cache["content_hash"]}
    summary = probe["summary"]
    if summary["checkpoint_hash"] != common["checkpoint_hash"] or summary["budget"] != _probe_config(config):
        raise ValueError("Teacher state or fixed probe budget changed")
    selected_path = Path(summary["selected_probe_path"])
    selected = torch.load(selected_path, map_location="cpu", weights_only=True)
    if selected["probe_hash"] != summary["probe_hash"] or encoder_state_hash(selected["state_dict"]) != summary["probe_hash"]:
        raise ValueError("Selected teacher probe state changed")
    if selected["identity"]["train_cache"] != caches["train"]["content_hash"] or selected["identity"]["dev_cache"] != caches["development"]["content_hash"]:
        raise ValueError("Selected teacher probe cache identity changed")
    stats = fit_standardization(caches["train"]["features"], summary["budget"]["standardize"])
    if any(not torch.equal(stats[key], selected["standardization"][key]) for key in ("mean", "std")):
        raise ValueError("Teacher standardization does not equal training-only statistics")
    classifier = nn.Linear(selected["input_dim"], 101)
    classifier.load_state_dict(selected["state_dict"])
    classifier.eval()
    with torch.inference_mode():
        features = standardize(caches["development"]["features"], stats)
        logits = torch.cat([classifier(x) for x in features.split(summary["budget"]["batch_size"])])
        predictions = logits.argmax(dim=1)
        top5 = logits.topk(5, dim=1).indices
    metrics = classification_metrics(logits, caches["development"]["labels"])
    if any(abs(metrics[key] - summary["metrics"][key]) > 1e-7 for key in ("top1", "top5", "loss")):
        raise ValueError("Teacher probe reconstructed metrics differ")
    rows = probe["predictions"]
    if [r["sample_id"] for r in rows] != list(bundle["dev_ids"]):
        raise ValueError("Teacher prediction order differs from development split")
    for index, row in enumerate(rows):
        label = int(caches["development"]["labels"][index])
        if int(row["label"]) != label or int(row["correct_top1"]) != int(predictions[index] == label) or int(row["correct_top5"]) != int((top5[index] == label).any()):
            raise ValueError("Teacher predictions do not reconstruct from saved features and selected probe")
    probe = _fixed_final_epoch_probe(probe, caches, Path(summary["resume_manifest"]).parent)
    return probe, {"summary_path": str(summary_path), "summary_sha256": entry["sha256"],
        "selected_probe_path": str(selected_path), "selected_probe_sha256": digest_file(selected_path),
        "cache_evidence": cache_evidence, "reconstructed_predictions": len(rows),
        "train_only_standardization_verified": True, "encoder_state_hash": common["checkpoint_hash"]}

def _ema_evidence(source, kind):
    rows = list(csv.DictReader((source / (kind + "_training_metrics.csv")).open()))
    if [int(row["successful_updates"]) for row in rows] != list(range(1, 501)):
        raise ValueError("EMA coefficient requires exactly the complete 500-update history")
    momenta = [float(row["ema_momentum"]) for row in rows]
    if not all(0 < value <= 1 for value in momenta):
        raise ValueError("Invalid saved EMA momentum")
    return {"initial_parameter_coefficient": math.exp(sum(math.log(value) for value in momenta)),
            "updates": len(momenta), "first_momentum": momenta[0], "last_momentum": momenta[-1],
            "history_sha256": digest_file(source / (kind + "_training_metrics.csv")),
            "interpretation": "Coefficient in the affine parameter recurrence, not a percentage of random representation or proof of teacher lag."}


In [ ]:
def run_saved_encoder_diagnostics(root, artifact_dir, deadline_unix):
    """Run fixed-epoch-20 online frozen probes against verified epoch-20 teacher probes.

    Caller grants exclusive GPU 0 use and enforces its outer hard deadline. No
    training or heldout data evaluation occurs. All reusable identities are exact.
    """
    root, artifact = Path(root).resolve(), Path(artifact_dir).resolve()
    artifact.relative_to(root)
    if not artifact.name or artifact.name in (".", ".."):
        raise ValueError("A named experiment artifact directory is required")
    folder = artifact / "saved_encoder_diagnostics"
    folder.mkdir(parents=True, exist_ok=True)
    with run_writer(folder):
        return _run_saved_encoder_diagnostics(root, artifact, folder, float(deadline_unix))

def _run_saved_encoder_diagnostics(root, artifact, folder, deadline):
    _deadline_check(deadline)
    started = time.time()
    source = root / "artifacts/notebook_03" / SOURCE_RUN
    config = json.loads((source / "resolved_config.json").read_text())
    configure_environment(config)
    records = {}
    for kind in DIAGNOSTIC_KINDS:
        checkpoint_dir = root / "checkpoints" / SOURCE_RUN / kind
        manifest = validate_training_manifest(checkpoint_dir)
        if int(manifest["successful_updates"]) != 500 or int(manifest["attempted_updates"]) != 500:
            raise ValueError("Diagnostic requires the completed 500-update source checkpoint")
        path = checkpoint_dir / manifest["latest"]
        records[kind] = {"path": str(path), "sha256": digest_file(path), "updates": 500}
    identity = {"schema": "saved-online-teacher-diagnostic/v1", "source_run_id": SOURCE_RUN,
        "source_checkpoints": records, "resolved_config_sha256": digest_file(source / "resolved_config.json"),
        "source_evaluation_manifest_sha256": digest_file(source / "semantic_evaluation_manifest.json"),
        "source_definition_hashes": definition_hashes(root), "diagnostic_source_hash": _diagnostic_source_hash(root),
        "decision_rule": DECISION_RULE, "probe_budget": _probe_config(config),
        "original_deadline_unix": deadline, "heldout_status": "NOT_RUN"}
    identity_hash = stable_digest(identity)
    protocol_path = folder / "protocol.json"
    if protocol_path.exists():
        if json.loads(protocol_path.read_text())["identity_hash"] != identity_hash:
            raise ValueError("Diagnostic inputs, source, budget, or deadline changed; exact resume refused")
    else:
        atomic_json(protocol_path, {"identity_hash": identity_hash, "identity": identity,
            "registered_unix": time.time(), "registered_before_online_feature_extraction": True})
    manifest_path = folder / "manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        if manifest["identity_hash"] != identity_hash:
            raise ValueError("Diagnostic manifest identity changed")
    else:
        manifest = {"identity_hash": identity_hash, "completed": {}, "started_unix": started}
        atomic_json(manifest_path, manifest)
    bundle = reuse_food_bundle(root, config, folder, verify_images=True)
    _deadline_check(deadline)
    if len(bundle["train_ids"]) != 10100 or len(bundle["dev_ids"]) != 2020:
        raise ValueError("Diagnostic split size changed")
    checkpoint_dir = root / "checkpoints" / artifact.name / "saved_encoder_probes"
    probes, teacher_evidence, online_evidence, ema, contrasts = {}, {}, {}, {}, {}
    for kind in DIAGNOSTIC_KINDS:
        _deadline_check(deadline, reserve=5)
        model = encoder = None
        try:
            _release_semantic(config["device"])
            with isolated_rng(int(config["seeds"]["model_init"])):
                model = _build_semantic_comparator(config, kind)
            payload = torch.load(records[kind]["path"], map_location="cpu", weights_only=False)
            model.load_state_dict(payload["model"])
            del payload
            teacher, evidence = _load_verified_teacher(root, source, config, bundle, kind, model.target)
            probes[kind + "_teacher"] = teacher
            teacher_evidence[kind] = evidence
            ema[kind] = _ema_evidence(source, kind)
            encoder = model.context.cpu().eval()
            model = None
            online_hash = encoder_state_hash(encoder)
            model_id = kind + "_online"
            summary_path = folder / (model_id + "_probe.json")
            if model_id in manifest["completed"]:
                if digest_file(summary_path) != manifest["completed"][model_id]["sha256"]:
                    raise ValueError("Completed online probe summary changed")
                saved = json.loads(summary_path.read_text())
                if saved["identity_hash"] != identity_hash or saved["encoder_state_hash"] != online_hash:
                    raise ValueError("Completed online probe identity changed")
                probes[model_id] = saved["probe"]
                online_evidence[kind] = saved["evidence"]
            else:
                common = {"dataset_revision": bundle["dataset"].fingerprint,
                    "checkpoint_hash": online_hash, "encoder_kind": kind,
                    "preprocessing": preprocessing_descriptor(config),
                    "pooling": "mean_final_layernorm_patches",
                    "normalization": "encoder final affine LayerNorm then patch mean; train-only per-feature probe standardization"}
                caches, cache_evidence = {}, {}
                for split, ids in (("train", bundle["train_ids"]), ("development", bundle["dev_ids"])):
                    _deadline_check(deadline, reserve=2)
                    dataset = _DeadlineDataset(EvaluationDataset(bundle["dataset"], ids, config), deadline)
                    print(f"Saved encoder diagnostic: {model_id}, {split}, {len(ids)} images", flush=True)
                    cache = extract_features(encoder, dataset, folder / "features" / model_id,
                        {**common, "sample_ids": list(ids)}, batch_size=int(config["evaluation"]["batch_size"]), device=config["device"])
                    caches[split] = cache
                    cache_evidence[split] = {"path": cache["path"], "sha256": digest_file(cache["path"]),
                        "content_hash": cache["content_hash"], "reused": cache["reused"]}
                encoder.cpu()
                _release_semantic(config["device"])
                result = _diagnostic_probe(caches["train"], caches["development"], checkpoint_dir,
                    config, model_id, encoder, deadline)
                probe = {"summary": result["summary"], "metrics": result["metrics"], "predictions": result["predictions"]}
                probe = _fixed_final_epoch_probe(probe, caches, checkpoint_dir / model_id)
                probes[model_id] = probe
                selected_path = Path(result["summary"]["selected_probe_path"])
                online_evidence[kind] = {"cache_evidence": cache_evidence, "encoder_state_hash": online_hash,
                    "selected_probe_path": str(selected_path), "selected_probe_sha256": digest_file(selected_path),
                    "encoder_choice": "online_context", "source_checkpoint": records[kind]}
                write_csv(folder / (model_id + "_development.csv"), probe["predictions"])
                atomic_json(summary_path, {"identity_hash": identity_hash, "encoder_state_hash": online_hash,
                    "probe": probe, "evidence": online_evidence[kind]})
                manifest["completed"][model_id] = {"path": str(summary_path), "sha256": digest_file(summary_path)}
                atomic_json(manifest_path, manifest)
            online = probes[model_id]
            if online["summary"]["initialization_hash"] != teacher["summary"]["initialization_hash"] or online["summary"]["budget"] != teacher["summary"]["budget"]:
                raise ValueError("Online and teacher probes do not share initialization and budget")
            orders = lambda probe: [(row["epoch"], row["split"], row["permutation_hash"]) for row in probe["metrics"]]
            if orders(online) != orders(teacher):
                raise ValueError("Online and teacher probes have different training orders")
            contrasts[kind] = paired_bootstrap(online["predictions"], teacher["predictions"],
                repeats=int(config["probe"]["bootstrap_repeats"]), seed=int(config["seeds"]["bootstrap"]), metric="correct_top1")
            print(f"{kind}: online {100*online['summary']['metrics']['top1']:.2f}%, teacher {100*teacher['summary']['metrics']['top1']:.2f}%", flush=True)
        finally:
            model = encoder = None
            _release_semantic(config["device"])
    triggered = any(value["delta_percentage_points"] >= DECISION_RULE["threshold_percentage_points"] for value in contrasts.values())
    result = {"schema": "saved-online-teacher-diagnostic/v1", "status": "PASS", "identity_hash": identity_hash,
        "source_run_id": SOURCE_RUN, "probes": {key: value["summary"] for key, value in probes.items()},
        "online_minus_teacher_top1": contrasts, "ema_evidence": ema,
        "teacher_evidence": teacher_evidence, "online_evidence": online_evidence,
        "decision_rule": DECISION_RULE, "ema_triggered": triggered,
        "selected_factor": DECISION_RULE["if_triggered"] if triggered else DECISION_RULE["otherwise"],
        "dataset": {"train_examples": 10100, "development_examples": 2020, "official_test": "NOT_RUN",
            "train_ids_hash": stable_digest(bundle["train_ids"]), "development_ids_hash": stable_digest(bundle["dev_ids"])},
        "probe_pairing_verified": {"initialization": True, "budget": True, "epoch_order": True, "train_only_standardization": True},
        "started_unix": manifest["started_unix"], "completed_unix": time.time(), "invocation_wall_seconds": time.time() - started,
        "uncertainty": "Paired development-image percentile bootstrap conditional on fixed encoders and fixed final probe epoch 20. Excludes training-seed and diagnostic-driven factor-selection uncertainty. Diagnostic outcome selects one future factor; it does not prove the factor will improve learning."}
    atomic_json(folder / "result.json", result)
    lines = ["# Saved encoder diagnostic", "", "Source: `" + SOURCE_RUN + "`. No encoder retraining. Both probes use fixed final epoch 20. Official test: **NOT_RUN**.", "",
        "| Architecture | Online top-1 | EMA teacher top-1 | Online minus teacher (pp) | Paired 95% interval (pp) |", "|---|---:|---:|---:|---:|"]
    for kind in DIAGNOSTIC_KINDS:
        delta = contrasts[kind]; lo, hi = delta["ci95_percentage_points"]
        lines.append(f"| {kind} | {100*probes[kind+'_online']['summary']['metrics']['top1']:.2f}% | {100*probes[kind+'_teacher']['summary']['metrics']['top1']:.2f}% | {delta['delta_percentage_points']:+.2f} | [{lo:+.2f}, {hi:+.2f}] |")
    lines += ["", "Predeclared next factor: **" + result["selected_factor"] + "**.", "", result["uncertainty"], "",
        "EMA initialization coefficient: " + ", ".join(f"{kind} {ema[kind]['initial_parameter_coefficient']:.6f}" for kind in DIAGNOSTIC_KINDS) + ". This is a parameter recurrence coefficient, not a percentage of random representation."]
    (folder / "report.md").write_text("\n".join(lines) + "\n")
    return result


In [ ]:
assert callable(run_saved_encoder_diagnostics)
print("Saved encoder diagnostic definitions loaded; the Notebook 04 controller owns GPU execution.")